[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/25_time_series/03_arima_family.ipynb)

# Bài 03 — Họ ARIMA

**🎯 Mục tiêu**
- Mô hình **AR(p)** và **MA(q)**; nhận diện bằng **ACF / PACF** (PACF cài đặt from scratch).
- Ước lượng AR bằng **bình phương tối thiểu** from scratch; chọn bậc bằng **AIC**; dự báo đệ quy và khoảng dự báo bằng mô phỏng.
- **ARIMA / SARIMA**: sai phân thường và sai phân mùa vụ; thêm biến ngoại sinh (**ARX**) — khuyến mãi, ngày lễ biết trước.
- Chẩn đoán phần dư (kiểm định Ljung–Box from scratch); giới hạn của họ ARIMA.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/25_time_series

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import chi2

from ml_course.visualization import set_style

set_style()
rng = np.random.default_rng(0)
try:
    from statsmodels.stats.diagnostic import acorr_ljungbox
    from statsmodels.tsa.ar_model import AutoReg
    from statsmodels.tsa.stattools import pacf as sm_pacf
    HAS_SM = True
except ImportError:                                                     # statsmodels là tuỳ chọn: pip install -e ".[stats]"
    HAS_SM = False

## 1. AR và MA

- **AR(p)** — tự hồi quy: $y_t = c + \phi_1 y_{t-1} + \dots + \phi_p y_{t-p} + \varepsilon_t$. Giá trị hiện tại là tổ hợp tuyến tính của quá khứ. Dừng khi các nghiệm của $1 - \phi_1 z - \dots - \phi_p z^p$ nằm ngoài đường tròn đơn vị (AR(1): $|\phi_1| < 1$).
- **MA(q)** — trung bình trượt của **cú sốc**: $y_t = c + \varepsilon_t + \theta_1\varepsilon_{t-1} + \dots + \theta_q\varepsilon_{t-q}$. Một cú sốc chỉ ảnh hưởng $q$ bước.

**PACF** (tự tương quan riêng phần) tại độ trễ $k$: tương quan giữa $y_t$ và $y_{t-k}$ **sau khi loại ảnh hưởng** của $y_{t-1}, \dots, y_{t-k+1}$ = hệ số của $y_{t-k}$ trong hồi quy $y_t$ trên $k$ trễ đầu tiên.

In [ ]:
def acf(y, max_lag):
    y = np.asarray(y, float) - np.mean(y)
    return np.array([1.0] + [np.sum(y[k:] * y[:-k]) / np.sum(y**2) for k in range(1, max_lag + 1)])


def lag_matrix(y, p):
    """Ma trận [1, y_{t-1}, ..., y_{t-p}] cho t = p..n-1."""
    y = np.asarray(y, float)
    return np.column_stack([np.ones(len(y) - p)] + [y[p - k:len(y) - k] for k in range(1, p + 1)])


def pacf(y, max_lag):
    out = [1.0]
    for k in range(1, max_lag + 1):
        coef, *_ = np.linalg.lstsq(lag_matrix(y, k), np.asarray(y, float)[k:], rcond=None)
        out.append(coef[-1])                                            # hệ số của trễ xa nhất
    return np.array(out)


def simulate(phi=(), theta=(), n=600, seed=0):
    r = np.random.default_rng(seed)
    e = r.standard_normal(n + 100)
    y = np.zeros(n + 100)
    for t in range(len(y)):
        y[t] = e[t] + sum(p * y[t - 1 - i] for i, p in enumerate(phi) if t - 1 - i >= 0) \
               + sum(q * e[t - 1 - i] for i, q in enumerate(theta) if t - 1 - i >= 0)
    return y[100:]                                                      # bỏ giai đoạn khởi động


series = {"AR(1), φ = 0.8": simulate(phi=[0.8]), "AR(2), φ = (0.5, 0.3)": simulate(phi=[0.5, 0.3]),
          "MA(1), θ = 0.8": simulate(theta=[0.8]), "ARMA(1,1)": simulate(phi=[0.7], theta=[0.5])}
fig, axes = plt.subplots(2, 4, figsize=(22, 7))
band = 1.96 / np.sqrt(600)
for j, (name, y) in enumerate(series.items()):
    for i, (fn, label) in enumerate([(acf, "ACF"), (pacf, "PACF")]):
        axes[i, j].bar(range(16), fn(y, 15), width=0.4)
        axes[i, j].axhspan(-band, band, color="gray", alpha=0.2)
        axes[i, j].set_title(f"{label} — {name}", fontsize=10)
plt.tight_layout()
plt.show()
if HAS_SM:
    print("PACF tự cài đặt khớp statsmodels (method='ols'):", np.allclose(pacf(series["AR(2), φ = (0.5, 0.3)"], 10),
                                                                       sm_pacf(series["AR(2), φ = (0.5, 0.3)"], nlags=10, method="ols")))

| Mô hình | ACF | PACF |
|---|---|---|
| AR(p) | giảm dần (mũ/dao động) | **cắt** sau độ trễ $p$ |
| MA(q) | **cắt** sau độ trễ $q$ | giảm dần |
| ARMA(p, q) | giảm dần | giảm dần |

Đây là quy tắc kinh điển của Box–Jenkins; thực tế thường kết hợp với AIC vì biểu đồ mẫu nhiễu.

## 2. Ước lượng AR bằng bình phương tối thiểu

Hồi quy $y_t$ trên $[1, y_{t-1}, \dots, y_{t-p}]$ — chính là hồi quy tuyến tính (chương 06) với feature là các trễ. Chọn $p$ bằng $\text{AIC} = n\log(\text{SSE}/n) + 2k$ ($k$ = số tham số):

In [ ]:
def fit_ar(y, p, X=None):
    A = lag_matrix(y, p)
    if X is not None:
        A = np.column_stack([A, np.asarray(X)[p:]])
    target = np.asarray(y, float)[p:]
    coef, *_ = np.linalg.lstsq(A, target, rcond=None)
    resid = target - A @ coef
    aic = len(resid) * np.log(np.mean(resid**2)) + 2 * len(coef)
    return coef, resid, aic


y_ar2 = series["AR(2), φ = (0.5, 0.3)"]
for p in range(1, 7):
    coef, resid, aic = fit_ar(y_ar2, p)
    print(f"p = {p}: AIC = {aic:8.2f} | hệ số φ = {np.round(coef[1:], 3)}")
if HAS_SM:
    print("AutoReg(statsmodels, p=2):", np.round(AutoReg(y_ar2, lags=2).fit().params[1:], 3))

AIC nhỏ nhất ở $p = 2$ và hệ số ước lượng gần $(0.5, 0.3)$ thật. (MA cần ước lượng phi tuyến — cú sốc $\varepsilon_t$ không quan sát được — thường bằng MLE qua Kalman filter; thư viện làm việc này.)

## 3. Dự báo đệ quy và khoảng dự báo

Dự báo $h$ bước: dùng chính dự báo làm "quá khứ" cho bước sau. Với AR dừng, dự báo **quay về trung bình** và độ bất định tiến tới phương sai không điều kiện. Khoảng dự báo bằng **mô phỏng**: sinh nhiều đường tương lai với sai số lấy mẫu lại từ phần dư (bootstrap).

In [ ]:
def ar_forecast(history, coef, h):
    p = len(coef) - 1
    hist = list(history[-p:])
    out = []
    for _ in range(h):
        nxt = coef[0] + np.dot(coef[1:], hist[::-1][:p])
        out.append(nxt)
        hist.append(nxt)
    return np.array(out)


def ar_simulate_paths(history, coef, resid, h, n_paths=2000, seed=0):
    r = np.random.default_rng(seed)
    p = len(coef) - 1
    paths = np.empty((n_paths, h))
    for i in range(n_paths):
        hist = list(history[-p:])
        for k in range(h):
            nxt = coef[0] + np.dot(coef[1:], hist[::-1][:p]) + r.choice(resid)
            paths[i, k] = nxt
            hist.append(nxt)
    return paths


coef2, resid2, _ = fit_ar(y_ar2, 2)
point = ar_forecast(y_ar2, coef2, 40)
paths = ar_simulate_paths(y_ar2, coef2, resid2, 40)
lo, hi = np.percentile(paths, [2.5, 97.5], axis=0)
plt.figure(figsize=(13, 4))
plt.plot(range(-80, 0), y_ar2[-80:], color="k", lw=0.8)
plt.plot(range(40), point, color="r", label="dự báo điểm")
plt.fill_between(range(40), lo, hi, color="r", alpha=0.2, label="khoảng 95% (mô phỏng)")
plt.axhline(y_ar2.mean(), ls=":", color="gray", label="trung bình mẫu")
plt.legend()
plt.show()
print(f"độ rộng khoảng 95%: h=1 → {hi[0] - lo[0]:.2f} | h=40 → {hi[-1] - lo[-1]:.2f} (≈ 2·1.96·độ lệch chuẩn của chuỗi = {2 * 1.96 * y_ar2.std():.2f})")

## 4. ARIMA và SARIMA

- **ARIMA(p, d, q)**: lấy sai phân $d$ lần để chuỗi dừng, rồi mô hình ARMA(p, q).
- **SARIMA(p, d, q)(P, D, Q)$_m$**: thêm sai phân **mùa vụ** $D$ lần ($y_t - y_{t-m}$) và các thành phần AR/MA mùa vụ ở trễ $m, 2m, \dots$
- **ARIMAX / SARIMAX**: thêm biến ngoại sinh $\mathbf x_t$ (khuyến mãi, ngày lễ, giá) — cần **biết trước** giá trị tương lai của chúng.

Áp dụng cho log doanh số: sai phân mùa vụ $z_t = \log y_t - \log y_{t-7}$ khử mùa vụ tuần và phần lớn xu hướng.

In [ ]:
df = pd.read_csv("../../data/sample/sales_daily.csv", parse_dates=["date"]).set_index("date").asfreq("D")
sales = df["sales"].values
log_y = np.log(sales)
H, m = 91, 7
T = len(sales) - H
z = log_y[m:T] - log_y[:T - m]                                         # chỉ dùng dữ liệu train
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].plot(df.index[m:T], z, lw=0.5)
axes[0].set_title("z = log yₜ − log yₜ₋₇")
for ax, (fn, label) in zip(axes[1:], [(acf, "ACF"), (pacf, "PACF")]):
    ax.bar(range(29), fn(z, 28), width=0.4)
    ax.axhspan(-1.96 / np.sqrt(len(z)), 1.96 / np.sqrt(len(z)), color="gray", alpha=0.2)
    ax.set_title(f"{label} của z")
plt.tight_layout()
plt.show()

ACF của $z$ có đỉnh **âm** lớn tại độ trễ 7: dấu hiệu kinh điển của sai phân mùa vụ (một cú sốc ngày $t$ xuất hiện ở $z_t$ với dấu + và ở $z_{t+7}$ với dấu −) → cần thành phần **MA mùa vụ** $\Theta_1$ (SARIMA $(\cdot)(0,1,1)_7$). Mô hình AR thuần cần **nhiều trễ** (7, 14, ...) để xấp xỉ nó.

Thử AR(p) trên $z$, có/không có biến ngoại sinh (khuyến mãi, lễ — cũng lấy sai phân mùa vụ cho khớp), dự báo đệ quy 91 ngày rồi đảo ngược sai phân: $\log\hat y_t = \log\hat y_{t-7} + \hat z_t$.

In [ ]:
X_exog = np.column_stack([df["promo"].values[m:] - df["promo"].values[:-m],
                          df["is_holiday"].values[m:] - df["is_holiday"].values[:-m]]).astype(float)   # căn với z (t = 7..n−1)
mase_scale = np.mean(np.abs(sales[:T][m:] - sales[:T][:-m]))


def forecast_seasonal_ar(p, use_exog):
    X_tr = X_exog[:T - m] if use_exog else None
    coef, _, aic = fit_ar(z, p, X_tr)
    z_hist, log_hist = list(z), list(log_y[:T])
    for h in range(H):
        t = T + h
        z_hat = coef[0] + np.dot(coef[1:p + 1], z_hist[::-1][:p])
        if use_exog:
            z_hat += np.dot(coef[p + 1:], X_exog[t - m])                 # khuyến mãi/lễ tương lai đã biết trước
        z_hist.append(z_hat)
        log_hist.append(log_hist[t - m] + z_hat)
    pred = np.exp(np.array(log_hist[T:]))
    return aic, np.mean(np.abs(sales[T:] - pred)) / mase_scale, pred


rows, preds = [], {}
for use_exog in [False, True]:
    for p in [1, 2, 7, 14]:
        aic, mase, pred = forecast_seasonal_ar(p, use_exog)
        preds[(p, use_exog)] = pred
        rows.append({"mô hình": f"AR({p}) trên z" + (" + khuyến mãi/lễ" if use_exog else ""), "AIC (train)": aic, "MASE test": mase})
pd.DataFrame(rows).set_index("mô hình").round(3)

- Không có biến ngoại sinh, AR trên $z$ còn **kém** Holt–Winters ở bài 02 (MASE 0.84) — các đỉnh khuyến mãi trong lịch sử làm nhiễu ước lượng, và mô hình không biết ngày nào có khuyến mãi trong tương lai.
- Thêm khuyến mãi/lễ (đã biết trước theo lịch) → MASE giảm mạnh, tốt nhất đến lúc này. **Thông tin** đúng quan trọng hơn **thuật toán**.
- AIC (tính trên train) chọn cùng hướng với MASE test: nhiều trễ hơn (bắt cấu trúc trễ 7, 14) và có biến ngoại sinh.

In [ ]:
best_p, best_exog = 14, True
aic_b, mase_b, pred_b = forecast_seasonal_ar(best_p, best_exog)
idx_test = df.index[T:]
plt.figure(figsize=(14, 4))
plt.plot(df.index[T - 60:], sales[T - 60:], color="k", lw=0.8, label="thực tế")
plt.plot(idx_test, pred_b, color="r", label=f"AR(14) trên z + khuyến mãi/lễ (MASE {mase_b:.3f})")
plt.plot(idx_test, preds[(14, False)], color="C0", alpha=0.7, label="AR(14) trên z, không biến ngoại sinh")
plt.axvline(idx_test[0], color="gray", ls=":")
plt.legend()
plt.show()

## 5. Chẩn đoán phần dư: Ljung–Box

Nếu mô hình đã bắt hết cấu trúc, phần dư phải như **nhiễu trắng**. Kiểm định Ljung–Box với $h$ độ trễ:
$$Q = n(n + 2)\sum_{k=1}^{h}\frac{\hat\rho_k^2}{n - k} \sim \chi^2_{h - \text{số tham số ARMA}}$$
$H_0$: phần dư không tự tương quan. p-value nhỏ → còn cấu trúc chưa khai thác.

In [ ]:
def ljung_box(resid, h, n_params=0):
    n = len(resid)
    r = acf(resid, h)[1:]
    Q = n * (n + 2) * np.sum(r**2 / (n - np.arange(1, h + 1)))
    return Q, chi2.sf(Q, h - n_params)


for p, use_exog in [(1, False), (14, False), (14, True)]:
    _, resid, _ = fit_ar(z, p, X_exog[:T - m] if use_exog else None)
    Q, pval = ljung_box(resid, 21, n_params=p)
    print(f"AR({p:2d}) {'+ exog' if use_exog else '      '}: Ljung–Box Q(21) = {Q:7.1f} | p-value = {pval:.4f}")
if HAS_SM:
    _, resid1, _ = fit_ar(z, 1)
    print("statsmodels acorr_ljungbox AR(1), lag 21:", acorr_ljungbox(resid1, lags=[21], model_df=1).round(4).to_dict("records"))

AR(1) để lại nhiều tự tương quan (p-value ≈ 0). Thêm trễ (AR(14)) làm $Q$ giảm mạnh; biến ngoại sinh chỉ giảm thêm **rất ít** — chúng làm phần dư **nhỏ đi** (AIC giảm mạnh, MASE tốt hơn nhiều) chứ không làm phần dư "trắng" hơn: Ljung–Box đo **tự tương quan**, không đo độ lớn sai số. p-value vẫn ≈ 0 → còn cấu trúc chưa khai thác (thành phần MA mùa vụ, mùa vụ năm) — hướng của SARIMAX đầy đủ hoặc mô hình feature-based ở bài 04.

### Tham khảo: SARIMAX với statsmodels

Thư viện ước lượng đầy đủ SARIMA (gồm cả MA, bằng MLE + Kalman filter). Đoạn code tham khảo dưới đây **không chạy** trong notebook (một số môi trường chặn phần biên dịch của `statsmodels.tsa.statespace`):

```python
from statsmodels.tsa.statespace.sarimax import SARIMAX

exog = df[["promo", "is_holiday"]].astype(float)
model = SARIMAX(np.log(df["sales"].iloc[:T]), exog=exog.iloc[:T],
                order=(1, 0, 1), seasonal_order=(0, 1, 1, 7)).fit(disp=False)
fc = model.get_forecast(steps=H, exog=exog.iloc[T:])
pred = np.exp(fc.predicted_mean)              # dự báo điểm
ci = np.exp(fc.conf_int(alpha=0.05))          # khoảng 95%
print(model.summary())                        # hệ số, AIC, Ljung–Box...
```

Chọn bậc tự động: `pmdarima.auto_arima` hoặc `statsforecast.AutoARIMA` (tìm theo AIC).

## 6. Giới hạn của họ ARIMA

| Ưu điểm | Nhược điểm |
|---|---|
| nền tảng thống kê, khoảng dự báo, diễn giải được | tuyến tính; một chuỗi một mô hình (không học chung nhiều chuỗi) |
| tốt với chuỗi ngắn, cấu trúc tự tương quan rõ | khó với nhiều mùa vụ (tuần + năm), mùa vụ dài ($m = 365$) |
| ARIMAX thêm biến ngoại sinh | biến ngoại sinh phải biết trước; chọn bậc tốn công |

## 🧠 Kiểm tra nhanh

1. ACF/PACF của AR(2) và MA(1) trông thế nào?
2. Vì sao ước lượng AR chỉ là hồi quy tuyến tính? MA thì sao?
3. Vì sao dự báo AR dừng quay về trung bình khi $h$ lớn?
4. Đỉnh âm ở trễ 7 trong ACF sau sai phân mùa vụ gợi ý thành phần nào?
5. Ljung–Box kiểm tra điều gì?

## 📌 Tóm tắt

- AR: hồi quy trên trễ (PACF cắt sau $p$); MA: cú sốc ngắn hạn (ACF cắt sau $q$); chọn bậc bằng ACF/PACF + AIC.
- ARIMA = sai phân + ARMA; SARIMA thêm mùa vụ; ARIMAX thêm biến ngoại sinh biết trước — ở đây cải thiện lớn nhất.
- Phần dư phải như nhiễu trắng (Ljung–Box); giới hạn: tuyến tính, từng chuỗi, mùa vụ dài.

## 📚 Tài liệu tham khảo

- Box, Jenkins, Reinsel & Ljung (2015) — *Time Series Analysis: Forecasting and Control* (5th ed.)
- Hyndman & Athanasopoulos — *Forecasting: Principles and Practice* (3rd ed.), ch. 9–10
- Ljung & Box (1978) — *On a Measure of Lack of Fit in Time Series Models*